# 03 · Model ComparisonTrain every configured model on identical walk-forward folds and inspect the winner.

In [ ]:
import sys, warningssys.path.insert(0, "..")warnings.filterwarnings("ignore")from src.config import Configfrom src.data.loader import load_datasetfrom src.data.validation import validatefrom src.data.cleaning import cleanconfig = Config.load(config_path="../config/config.yaml", model_params_path="../config/model_params.yaml")config.root = config.root  # notebooks run from notebooks/raw = load_dataset("../data/sample/sample_store_sales.csv", config, preset="favorita")report = validate(raw, config)frame, stats = clean(raw, config)print(report.passed, frame.shape, stats)

In [ ]:
from src.features.builder import FeatureBuilderbuilder = FeatureBuilder(config)features = builder.drop_warmup(builder.fit_transform(frame))names = builder.spec.feature_names

## Train and compare

In [ ]:
from src.models.train import train_models, comparison_table, best_model_nameresults = train_models(features, names, config, models=["naive", "lightgbm", "xgboost"])table = comparison_table(results)table

## Fold-level detail for the winner

In [ ]:
import pandas as pdbest = best_model_name(table)pd.DataFrame(results[best].fold_metrics)

## Actual vs predicted

In [ ]:
import matplotlib.pyplot as pltpreds = results[best].predictions.groupby("date")[["actual", "prediction"]].sum()preds.plot(figsize=(12, 4), title=f"{best}: actual vs predicted");

## SHAP explanation

In [ ]:
from src.explain import shap_explainerfrom src.models.train import fit_final_modelmodel = fit_final_model(features, names, best, config)exp = shap_explainer.explain(model, features[names], max_samples=1000)exp.global_importance(15) if exp else "model does not support SHAP"

## Forecast and insights

In [ ]:
from src.forecast.recursive import forecast, forecast_summaryfrom src.insights.engine import InsightEnginefc = forecast(model, frame, builder, horizon=30, config=config)print(forecast_summary(fc, frame))InsightEngine(frame, fc, exp.global_importance() if exp else None).to_frame()